In [1]:
from PySide6.QtCore import (QCoreApplication, QDate, QDateTime, QLocale,
    QMetaObject, QObject, QPoint, QRect, QTimer, QTimer,
    QSize, QTime, QUrl, Qt)
from PySide6.QtGui import (QBrush, QColor, QConicalGradient, QCursor,
    QFont, QFontDatabase, QGradient, QIcon, QIntValidator,
    QImage, QKeySequence, QLinearGradient, QPainter,
    QPalette, QPixmap, QRadialGradient, QTransform)
from PySide6.QtWidgets import (QApplication, QComboBox, QFrame, QHBoxLayout,
    QLabel, QLineEdit, QMainWindow, QPushButton, QFileDialog, QDialog, QMessageBox,
    QSizePolicy, QWidget)
import sys
import time
import datetime
import json
from pathlib import Path

import cv2
from PIL import ImageFont, ImageDraw, Image
from ui_yolo26 import *
from yolo import *
from alpr_kor import *
from rknnlite.api import RKNNLite

TASKS = ["detect", "pose", "seg", "obb", "depth"]
DATASETS = ["COCO", "FIRE", "CRACK", "LICENSE", "GARBAGE", "PCB"]

class MyWindow(QMainWindow, Ui_MainWindow) :
    def __init__(self) :
        super().__init__()
        self.IMG_PATH = None
        self.RK3588_RKNN_MODEL = None
        self.task = TASKS[0]
        self.DATASET = COCO
        self.input_size = 640
        self.dir_path = str(Path.cwd())
        self.yolo = None
        self.setupUi(self)
        self.lineEdit.setValidator(QIntValidator(320, 640, self))

    def model_select(self):
        dialog = QFileDialog(self)
        dialog.setWindowTitle("모델 열기")
        dialog.setNameFilter("Models (*.rknn)")
        dialog.setFileMode(QFileDialog.ExistingFile)
        dialog.setOption(QFileDialog.DontUseNativeDialog, True)  # Qt 다이얼로그 사용
        dialog.resize(400, 300)  # 다이얼로그 크기 축소
        # 시작 디렉토리 지정
        dialog.setDirectory(self.dir_path)

        # 커서 위치 (메뉴 클릭 위치)
        cursor_pos = QCursor.pos()

        # 다이얼로그 띄운 뒤 위치 이동
        def position_and_exec():
            dialog.move(cursor_pos)
            dialog.exec()

        QTimer.singleShot(0, position_and_exec)

        # 선택 파일 처리
        def on_file_selected():
            selected_files = dialog.selectedFiles()
            if selected_files:
                file_path = selected_files[0]
                #self.load_image_from_path(file_path)
                print( file_path )
                self.RK3588_RKNN_MODEL = str(file_path)
                file_path = Path( file_path )
                self.pushButton_2.setText( file_path.name )
                """
                name = os.path.splitext(file_path.name)[0]   # LicensePlate-RK3588_320_i8
                parts = name.split("_")
                print(parts)
                if "320" in parts:
                    print("해상도 320")
                    self.lineEdit.setText(str(320))
                elif "640" in parts:
                    print("해상도 640")
                    self.lineEdit.setText(str(640))
                """
                parts = detect_imgsz_from_path(self.RK3588_RKNN_MODEL)
                self.lineEdit.setText(str(parts))
                
        dialog.fileSelected.connect(on_file_selected)

    def image_select(self):
        dialog = QFileDialog(self)
        dialog.setWindowTitle("이미지 열기")
        dialog.setNameFilter("Images (*.png *.jpg *.jpeg *.bmp *.gif)")
        dialog.setFileMode(QFileDialog.ExistingFile)
        dialog.setOption(QFileDialog.DontUseNativeDialog, True)  # Qt 다이얼로그 사용
        dialog.resize(400, 300)  # 다이얼로그 크기 축소
        # 시작 디렉토리 지정
        dialog.setDirectory(self.dir_path)

        # 커서 위치 (메뉴 클릭 위치)
        cursor_pos = QCursor.pos()

        # 다이얼로그 띄운 뒤 위치 이동
        def position_and_exec():
            dialog.move(cursor_pos)
            dialog.exec()

        QTimer.singleShot(0, position_and_exec)

        # 선택 파일 처리
        def on_file_selected():
            selected_files = dialog.selectedFiles()
            if selected_files:
                file_path = selected_files[0]
                #self.load_image_from_path(file_path)
                print( file_path )
                self.IMG_PATH = str(file_path)
                file_path = Path(file_path)
                self.pushButton_3.setText( file_path.name )
                self.load_image_from_path(self.IMG_PATH)

        dialog.fileSelected.connect(on_file_selected)

    def load_image_from_path(self, file_path):
        self.image = cv2.imread(file_path)
        image, ratio = letterbox(self.image, 640, 480, 0)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, ch = image_rgb.shape
        bytes_per_line = ch * w
        qimage = QImage(image_rgb.data, w, h, bytes_per_line, QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(qimage)
        self.label_4.setPixmap(pixmap)

    def task_select(self, index):
        #print(self.comboBox.itemText(index), index)
        self.task = TASKS[index]
        print( self.task)

    def dataset_select(self, index):
        #print(self.comboBox_3.itemText(index), index)
        self.DATASET = index
        print( self.DATASET )

    def cvimg_to_qpixmap(self, cv_img):
        image_rgb = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)
        height, width, channel = image_rgb.shape
        bytes_per_line = channel * width
        qimg = QImage(image_rgb.data, width, height, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimg)
    
    def inference(self):
        if self.RK3588_RKNN_MODEL is None:
            #print("RKNN Model not select!!")

            msg = QMessageBox(self)
            msg.setWindowTitle(" ")
            msg.setText("<b>경고</b>")
            msg.setInformativeText("모델이 선택되지 않았습니다.")
            msg.adjustSize()
            parent_center = self.geometry().center()
            msg_rect = msg.frameGeometry()
            msg_rect.moveCenter(parent_center)

            msg.move(msg_rect.topLeft())
            msg.setIcon(QMessageBox.Warning)
            msg.setStandardButtons(QMessageBox.Ok)
            msg.exec()
        elif self.IMG_PATH is None:
            #print("Image not select!!")

            msg = QMessageBox(self)
            msg.setWindowTitle(" ")
            msg.setText("<b>경고</b>")
            msg.setInformativeText("이미지가 선택되지 않았습니다.")
            msg.adjustSize()
            parent_center = self.geometry().center()
            msg_rect = msg.frameGeometry()
            msg_rect.moveCenter(parent_center)

            msg.move(msg_rect.topLeft())
            msg.setIcon(QMessageBox.Warning)
            msg.setStandardButtons(QMessageBox.Ok)
            msg.exec()
        else:
            S_IOU_THRESH = float(self.O_SCORE_VAL.text())
            DS_OBJ_THRESH = float(self.DS_OBJ_VAL.text())
            O_NMS_THRESH = float(self.O_NMS_VAL.text())
            P_OBJ_THRESH = float(self.P_OBJ_VAL.text())
            O_SCORE_THRESH = float(self.O_SCORE_VAL.text())

            self.load_image_from_path(self.IMG_PATH)
            if self.yolo is not None:
                self.yolo.release()
            if self.task == "detect" :
                self.yolo = Yolo( "detect", 
                                self.RK3588_RKNN_MODEL, 
                                DATASET = self.DATASET,
                                S_IOU_THRESH = S_IOU_THRESH,
                                DS_OBJ_THRESH = DS_OBJ_THRESH,
                                O_NMS_THRESH = O_NMS_THRESH,
                                O_SCORE_THRESH = O_SCORE_THRESH,
                                P_OBJ_THRESH = P_OBJ_THRESH,
                                )

                #boxes, classes, scores = self.yolo.detect(self.image)
                objects = self.yolo.detect(self.image)
                if objects is not None:
                    self.yolo.detect.draw(objects)
                    if self.DATASET == LICENSE:
                        print("LICENSE")
                        alpr = TFliteDemo('alpr_kor.tflite')
                        img_pil = Image.fromarray(self.yolo.detect.img_result)
                        font = ImageFont.truetype("./fonts/NanumGothicBold.ttf",20)
                        draw = ImageDraw.Draw(img_pil)

                        count = 0
                        for obj in objects:
                            left, top, right, bottom = obj['box']
                            plate_image = self.yolo.detect.img_org[top:bottom,left:right].copy()
                            just_image = just( plate_image )
                            #start_time = time.time()
                            new_label, conf = alpr.run(just_image)
                            if conf > 0.6:
                                print( new_label )
                                draw.text((5, 20 + 20*count ),f'#{count} 번호판(License) : {new_label}', font=font, fill=(0, 0, 255))
                                count += 1
                        self.yolo.detect.img_result = np.array(img_pil)
                    image, ratio = letterbox(self.yolo.detect.img_result, 640, 480, 0)
                    pixmap = self.cvimg_to_qpixmap(image)
                    self.label_4.setPixmap(pixmap)
                    self.lineEdit_7.setText(f'{self.yolo.detect.infertime:.4f}ms')
                else:
                    print( "Box is None.")
            elif self.task == "pose" :
                self.yolo = Yolo( "pose", 
                                self.RK3588_RKNN_MODEL, 
                                DATASET = self.DATASET,
                                S_IOU_THRESH = S_IOU_THRESH,
                                DS_OBJ_THRESH = DS_OBJ_THRESH,
                                O_NMS_THRESH = O_NMS_THRESH,
                                O_SCORE_THRESH = O_SCORE_THRESH,
                                P_OBJ_THRESH = P_OBJ_THRESH,
                                CALC_ANGLE=True
                                )
                objects = self.yolo.pose(self.image)
                if objects is not None:
                    self.yolo.pose.draw(objects)
                    image, ratio = letterbox(self.yolo.pose.img_result, 640, 480, 0)
                    pixmap = self.cvimg_to_qpixmap(image)
                    self.label_4.setPixmap(pixmap)
                    self.lineEdit_7.setText(f'{self.yolo.pose.infertime:.4f}ms')
                else:
                    print( "Box is None.")
            elif self.task == "seg" :
                self.yolo = Yolo( "seg", 
                                self.RK3588_RKNN_MODEL, 
                                DATASET = self.DATASET,
                                S_IOU_THRESH = S_IOU_THRESH,
                                DS_OBJ_THRESH = DS_OBJ_THRESH,
                                O_NMS_THRESH = O_NMS_THRESH,
                                O_SCORE_THRESH = O_SCORE_THRESH,
                                P_OBJ_THRESH = P_OBJ_THRESH,
                                )
                objects = self.yolo.seg(self.image)
                if objects is not None:
                    self.yolo.seg.draw(objects)
                    image, ratio = letterbox(self.yolo.seg.img_result, 640, 480, 0)
                    pixmap = self.cvimg_to_qpixmap(image)
                    self.label_4.setPixmap(pixmap)
                    self.lineEdit_7.setText(f'{self.yolo.seg.infertime:.4f}ms')
                else:
                    print( "Box is None.")
            elif self.task == "obb" :
                self.yolo = Yolo( "obb", 
                                self.RK3588_RKNN_MODEL, 
                                DATASET = self.DATASET,
                                S_IOU_THRESH = S_IOU_THRESH,
                                DS_OBJ_THRESH = DS_OBJ_THRESH,
                                O_NMS_THRESH = O_NMS_THRESH,
                                O_SCORE_THRESH = O_SCORE_THRESH,
                                P_OBJ_THRESH = P_OBJ_THRESH,
                                )
                predbox = self.yolo.obb(self.image)
                objects = self.yolo.obb(self.image)
                if objects is not None:
                    self.yolo.obb.draw(objects)
                    image, ratio = letterbox(self.yolo.obb.img_result, 640, 480, 0)
                    pixmap = self.cvimg_to_qpixmap(image)
                    self.label_4.setPixmap(pixmap)
                    self.lineEdit_7.setText(f'{self.yolo.obb.infertime:.4f}ms')
                else:
                    print( "Box is None.")
            elif self.task == "depth" :
                self.yolo = Yolo( "depth", 
                                self.RK3588_RKNN_MODEL
                                )
                self.yolo.depth(self.image)
                #objects = self.yolo.depth(self.image)
                image, ratio = letterbox(self.yolo.depth.img_result, 640, 480, 0)
                pixmap = self.cvimg_to_qpixmap(image)
                self.label_4.setPixmap(pixmap)
                self.lineEdit_7.setText(f'{self.yolo.depth.infertime:.4f}ms')

    def closeEvent(self, event):
        if self.yolo is not None:
            self.yolo.release()
        event.accept()  # 종료 계속 진행
        
if __name__ == "__main__":
    import sys
    if not QApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QApplication.instance()

    myWindow = MyWindow() 
    myWindow.show()
    sys.exit(app.exec())


depth
/home/darkice/ExtUSB/Work/rknn/rknn_examples/Yolo26_rknn/models/depth/yolo26n-depth_768x576.rknn
/home/darkice/ExtUSB/Work/rknn/rknn_examples/Yolo26_rknn/images/bus.jpg
YOLO : --> Load YOLO26 model
done
YOLO : --> Init runtime environment YOLO26
I RKNN: [13:34:05.170] RKNN Runtime Information, librknnrt version: 2.4.0 (b458df3b4a@2026-01-17T10:53:35)
I RKNN: [13:34:05.170] RKNN Driver Information, version: 0.9.8
I RKNN: [13:34:05.170] RKNN Model Information, version: 6, toolkit version: 2.3.2(compiler version: 2.3.2 (@2025-04-03T08:26:16)), target: RKNPU v2, target platform: rk3588, framework name: ONNX, framework layout: NCHW, model inference type: static_shape


W Query dynamic range failed. Ret code: RKNN_ERR_MODEL_INVALID. (If it is a static shape RKNN model, please ignore the above warning message.)
/home/darkice/ExtUSB/Work/rknn/rknn_examples/Yolo26_rknn/yolo26depth.py:292: UserWarning: Image aspect ratio (640:640) does not match model aspect ratio (576:768). Rect input would be 768x768 but model expects 768x576. Results will differ from ultralytics PT predict. Export a rect model with --imgsz 768x768 for best accuracy.
  return prepare_input_rect(image, self.imgsz, normalize=False)


W RKNN: [13:34:05.216] query RKNN_QUERY_INPUT_DYNAMIC_RANGE error, rknn model is static shape type, please export rknn with dynamic_shapes
done
RKNN model: /home/darkice/ExtUSB/Work/rknn/rknn_examples/Yolo26_rknn/models/depth/yolo26n-depth_768x576.rknn (imgsz=(768, 576), core=auto)


SystemExit: 0

/home/darkice/venv/pyside6rknn/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
